# GifGrep Agent

The GifGrep Agent is an AI-powered GIF search assistant that finds the perfect GIF for any moment. It searches both **Giphy** and **Tenor** — the two largest GIF libraries — and returns direct links, previews, and metadata, so you can quickly pick and share the right one.


In [2]:
!pip install -q aixplain requests

## Add your API keys

Get your aiXplain access key from the [Integrations](https://platform.aixplain.com/account/integrations) page.

Get your Giphy API key from the [Giphy Developer Portal](https://developers.giphy.com/dashboard/).

Get your Tenor API key from the [Google Cloud Console](https://console.cloud.google.com/) (Tenor API v2).

In [ ]:
AixplainAPIKey  = "<YOUR_AIXPLAIN_API_KEY>"  
GiphyKey   = "<YOUR_GIPHY_API_KEY>"         
TenorKey   = "<YOUR_TENOR_API_KEY>"

import os
os.environ["TEAM_API_KEY"]  = AixplainAPIKey
os.environ["GIPHY_API_KEY"] = GiphyKey
os.environ["TENOR_API_KEY"] = TenorKey


In [ ]:
import inspect
from aixplain import Aixplain

aix = Aixplain(AixplainAPIKey)


# What is the GifGrep Agent?

The goal of this agent is to make GIF discovery fast and conversational. Instead of manually searching Giphy or Tenor, you describe what you need in natural language and the agent finds the best matches across both platforms.

## Agent Architecture

The agent uses two custom script tools:

- **Giphy Search** — queries the Giphy API with keyword search, supporting content rating filters and returning direct GIF URLs, page links, dimensions, and file sizes.
- **Tenor Search** — queries the Tenor API (v2) and returns GIF URLs, preview links, and metadata from Tenor's library.

## Agent Workflow

1. **Understand the request** — extract the mood, context, or keywords from the user's message.
2. **Search both platforms** — run the query on Giphy and/or Tenor depending on the request.
3. **Recommend** — surface the top matches with direct links and describe each GIF so the user can pick.


In [4]:
def search_giphy(query: str, max_results: int = 10, rating: str = "g") -> str:
    """
    Searches Giphy for GIFs matching the query.

    Parameters:
        query (str): Search keywords, e.g. 'happy dog', 'office handshake', 'cat laser'.
        max_results (int): Number of results to return (1-25, default: 10).
        rating (str): Content rating filter — 'g' (default), 'pg', 'pg-13', or 'r'.

    Returns:
        str: Formatted list of GIF results with title, URL, direct GIF link, and dimensions.
    """
    import requests
    import os

    api_key = os.environ.get("GIPHY_API_KEY")
    if not api_key:
        return "Error: GIPHY_API_KEY environment variable is not set."

    max_results = max(1, min(max_results, 25))

    url = "https://api.giphy.com/v1/gifs/search"
    params = {
        "api_key": api_key,
        "q": query,
        "limit": max_results,
        "rating": rating,
        "lang": "en",
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.HTTPError as e:
        return f"HTTP error: {e}"
    except requests.exceptions.RequestException as e:
        return f"Request error: {e}"

    gifs = data.get("data", [])
    if not gifs:
        return f"No GIFs found for \"{query}\"."

    results = []
    for i, gif in enumerate(gifs, start=1):
        title = gif.get("title", "Untitled")
        page_url = gif.get("url", "")
        images = gif.get("images", {})
        original = images.get("original", {})
        gif_url = original.get("url", "")
        width = original.get("width", "?")
        height = original.get("height", "?")
        size_bytes = int(original.get("size", 0))
        size_kb = f"{size_bytes // 1024}KB" if size_bytes else "unknown size"
        downsized = images.get("downsized_medium", {}).get("url", gif_url)
        results.append(
            f"[{i}] {title}\n"
            f"    Page:     {page_url}\n"
            f"    GIF URL:  {gif_url}\n"
            f"    Preview:  {downsized}\n"
            f"    Size:     {width}x{height}px, {size_kb}"
        )

    return "\n\n".join(results)


In [5]:
def search_tenor(query: str, max_results: int = 10) -> str:
    """
    Searches Tenor for GIFs matching the query.

    Parameters:
        query (str): Search keywords, e.g. 'excited', 'thumbs up', 'mind blown'.
        max_results (int): Number of results to return (1-20, default: 10).

    Returns:
        str: Formatted list of GIF results with title, direct GIF link, and preview URL.
    """
    import requests
    import os

    api_key = os.environ.get("TENOR_API_KEY")
    if not api_key:
        return "Error: TENOR_API_KEY environment variable is not set."

    max_results = max(1, min(max_results, 20))

    url = "https://tenor.googleapis.com/v2/search"
    params = {
        "key": api_key,
        "q": query,
        "limit": max_results,
        "media_filter": "gif,tinygif",
        "contentfilter": "medium",
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.HTTPError as e:
        return f"HTTP error: {e}"
    except requests.exceptions.RequestException as e:
        return f"Request error: {e}"

    results_data = data.get("results", [])
    if not results_data:
        return f"No GIFs found on Tenor for \"{query}\"."

    results = []
    for i, item in enumerate(results_data, start=1):
        title = item.get("content_description", "Untitled")
        item_url = item.get("itemurl", "")
        media_formats = item.get("media_formats", {})
        gif = media_formats.get("gif", {})
        gif_url = gif.get("url", "")
        dims = gif.get("dims", [0, 0])
        width, height = dims[0], dims[1]
        size_bytes = gif.get("size", 0)
        size_kb = f"{size_bytes // 1024}KB" if size_bytes else "unknown size"
        preview = media_formats.get("tinygif", {}).get("url", gif_url)
        results.append(
            f"[{i}] {title}\n"
            f"    Page:    {item_url}\n"
            f"    GIF URL: {gif_url}\n"
            f"    Preview: {preview}\n"
            f"    Size:    {width}x{height}px, {size_kb}"
        )

    return "\n\n".join(results)


In [6]:
SCRIPT_INTEGRATION_ID = "688779d8bfb8e46c273982ca"

# Inject API keys directly into each tool's source — os.environ is not
# available in aiXplain's cloud tool runtime, so the keys must be embedded.
def _inject(fn, replacements):
    src = inspect.getsource(fn)
    for placeholder, value in replacements.items():
        src = src.replace(placeholder, f'"{value}"')
    return src

giphy_tool = aix.Tool(
    name="Giphy Search",
    integration=SCRIPT_INTEGRATION_ID,
    config={
        "code": _inject(search_giphy, {'os.environ.get("GIPHY_API_KEY")': GiphyKey}),
        "function_name": "search_giphy",
    },
)
giphy_tool.save()
print(f"Created: {giphy_tool.name}")

tenor_tool = aix.Tool(
    name="Tenor Search",
    integration=SCRIPT_INTEGRATION_ID,
    config={
        "code": _inject(search_tenor, {'os.environ.get("TENOR_API_KEY")': TenorKey}),
        "function_name": "search_tenor",
    },
)
tenor_tool.save()
print(f"Created: {tenor_tool.name}")


Created: Giphy Search
Created: Tenor Search


# Build Your Agent

To create an agent, define:

* A unique **name** and **description** for its purpose.
* The **tools** it will use — here, two custom script tools wrapping the Giphy and Tenor search APIs.
* The **instructions** that guide the agent's search and recommendation behaviour.

In [7]:
gifgrep_agent = aix.Agent(
    name="GifGrep Agent",
    description="Searches Giphy and Tenor to find the perfect GIF for any moment.",
    instructions="""
    You are a GIF search assistant with access to both Giphy and Tenor.

    When a user asks for a GIF:
    1. Extract the key mood, action, or concept they want (e.g. "excited", "facepalm", "thumbs up").
    2. Search Giphy first using the Giphy Search tool.
    3. If the user asks for more options or Giphy results are weak, also search Tenor.
    4. Present the top 3-5 results with:
       - A short description of the GIF
       - The direct GIF URL (for embedding)
       - The page URL (for sharing)
       - Dimensions and file size

    Tips:
    - Use simple, evocative keywords — short queries work better than long sentences.
    - If the first search returns nothing relevant, try a synonym or broader term.
    - For safe/work contexts use rating="g" (already the default).
    - When the user describes an emotion or reaction, translate it to a concrete search term
      (e.g. "when the code finally works" → "happy victory dance").
    """,
    tools=[giphy_tool, tenor_tool],
)
gifgrep_agent.save()
print(f"Agent created: {gifgrep_agent.id}")


Agent created: 69c44799228f69075764f182


# Run your Agent

To ensure your agent is performing as expected, run it using the `run` method with sample inputs. Analyze the outputs, verify their accuracy, and debug any issues by inspecting the agent's configurations, tools, and instructions.


In [8]:
result = gifgrep_agent.run("Find me a GIF of a happy dog")


In [9]:
print(result.data.output)


1. Friends Dogs GIF by MOODMAN
   - GIF URL: [View GIF](https://media4.giphy.com/media/UuebWyG4pts3rboawU/giphy.gif?cid=d71fffbai4f874ufzfdckt0uqtut7z4mk6xx099j958tnqky&ep=v1_gifs_search&rid=giphy.gif&ct=g)
   - Page URL: [View Page](https://giphy.com/gifs/cute-happy-dog-excited-UuebWyG4pts3rboawU)
   - Dimensions: 480x480px, File Size: 6484KB

2. Excited Aww GIF
   - GIF URL: [View GIF](https://media2.giphy.com/media/3ndAvMC5LFPNMCzq7m/giphy.gif?cid=d71fffbai4f874ufzfdckt0uqtut7z4mk6xx099j958tnqky&ep=v1_gifs_search&rid=giphy.gif&ct=g)
   - Page URL: [View Page](https://giphy.com/gifs/cute-aww-eyebleach-3ndAvMC5LFPNMCzq7m)
   - Dimensions: 384x480px, File Size: 5088KB

3. Happy Dog GIF
   - GIF URL: [View GIF](https://media4.giphy.com/media/QvBoMEcQ7DQXK/giphy.gif?cid=d71fffbai4f874ufzfdckt0uqtut7z4mk6xx099j958tnqky&ep=v1_gifs_search&rid=giphy.gif&ct=g)
   - Page URL: [View Page](https://giphy.com/gifs/lap-QvBoMEcQ7DQXK)
   - Dimensions: 221x221px, File Size: 189KB


In [10]:
result = gifgrep_agent.run("I need a reaction GIF for when someone's code finally passes all the tests")
print(result.data.output)


1. Happy Robot  
   - GIF URL: [Happy Robot GIF](https://media1.giphy.com/media/mIZ9rPeMKefm0/giphy.gif?cid=d71fffbajeh4tsrhm27tuncsojpo2u9b1qimz4bvh0b07h2g&ep=v1_gifs_search&rid=giphy.gif&ct=g)  
   - Page URL: [View on Giphy](https://giphy.com/gifs/dancing-happy-mIZ9rPeMKefm0)  
   - Dimensions: 287x400px  
   - File Size: 632KB  

2. Excited Let's Go  
   - GIF URL: [Excited Lets Go GIF](https://media2.giphy.com/media/clxfpkatJA7Tr7RWxc/giphy.gif?cid=d71fffbajeh4tsrhm27tuncsojpo2u9b1qimz4bvh0b07h2g&ep=v1_gifs_search&rid=giphy.gif&ct=g)  
   - Page URL: [View on Giphy](https://giphy.com/gifs/ZyptoPower-excited-happy-dance-celebration-clxfpkatJA7Tr7RWxc)  
   - Dimensions: 273x313px  
   - File Size: 511KB  

3. Victory Dancing  
   - GIF URL: [Victory Dancing GIF](https://media0.giphy.com/media/d4blihcFNkwE3fEI/giphy.gif?cid=d71fffbajeh4tsrhm27tuncsojpo2u9b1qimz4bvh0b07h2g&ep=v1_gifs_search&rid=giphy.gif&ct=g)  
   - Page URL: [View on Giphy](https://giphy.com/gifs/dancing-dab-dabbin

In [11]:
result = gifgrep_agent.run("Search both Giphy and Tenor for office handshake GIFs and compare the results")
print(result.data.output)


Giphy Results for 'office handshake':

1. Awkward Season 3 GIF by The Office
   - GIF URL: [Link](https://media2.giphy.com/media/ru7IH0oTmfIK6tRTId/giphy.gif?cid=d71fffba5av63ikh02jjbx6rvleutvvne7np2gji818remvw&ep=v1_gifs_search&rid=giphy.gif&ct=g)
   - Page URL: [Link](https://giphy.com/gifs/theoffice-nbc-the-office-tv-ru7IH0oTmfIK6tRTId)
   - Dimensions: 480x400px, File Size: 5731KB

2. Season 7 Nbc GIF by The Office
   - GIF URL: [Link](https://media1.giphy.com/media/79msfurZnTrTxzwOlL/giphy.gif?cid=d71fffba5av63ikh02jjbx6rvleutvvne7np2gji818remvw&ep=v1_gifs_search&rid=giphy.gif&ct=g)
   - Page URL: [Link](https://giphy.com/gifs/theoffice-79msfurZnTrTxzwOlL)
   - Dimensions: 480x400px, File Size: 6057KB

3. 80S Greeting GIF
   - GIF URL: [Link](https://media2.giphy.com/media/J4JSpIwM6y3Q6xnHgg/giphy.gif?cid=d71fffba5av63ikh02jjbx6rvleutvvne7np2gji818remvw&ep=v1_gifs_search&rid=giphy.gif&ct=g)
   - Page URL: [Link](https://giphy.com/gifs/office-computer-handshake-J4JSpIwM6y3Q6xnHgg)


In [12]:
result = gifgrep_agent.run(
    "Can you find something more dramatic?",
    session_id=result.data.session_id
)
print(result.data.output)


1. Awkward Season 3 GIF by The Office
   - GIF URL: [Link](https://media2.giphy.com/media/ru7IH0oTmfIK6tRTId/giphy.gif?cid=d71fffbac03pj0nz1le015w1t0z1rzcapo8rfdgf46ao9805&ep=v1_gifs_search&rid=giphy.gif&ct=g)
   - Page URL: [Link](https://giphy.com/gifs/theoffice-nbc-the-office-tv-ru7IH0oTmfIK6tRTId)
   - Dimensions: 480x400px, File Size: 5731KB

2. Drama Le Sigh GIF
   - GIF URL: [Link](https://media2.giphy.com/media/cwbvB0MXZfnry/giphy.gif?cid=d71fffbac03pj0nz1le015w1t0z1rzcapo8rfdgf46ao9805&ep=v1_gifs_search&rid=giphy.gif&ct=g)
   - Page URL: [Link](https://giphy.com/gifs/drama-cwbvB0MXZfnry)
   - Dimensions: 410x250px, File Size: 355KB

3. Scary Movie Officer Doofy GIF
   - GIF URL: [Link](https://media3.giphy.com/media/12GzK1jYCaVCV2/giphy.gif?cid=d71fffbac03pj0nz1le015w1t0z1rzcapo8rfdgf46ao9805&ep=v1_gifs_search&rid=giphy.gif&ct=g)
   - Page URL: [Link](https://giphy.com/gifs/12GzK1jYCaVCV2)
   - Dimensions: 427x272px, File Size: 384KB

4. Handshake Ryan GIF
   - GIF URL: [Link]

# Save the Agent

Once you are happy with your agent, save it to access the agent endpoints.


In [13]:
gifgrep_agent.save()


Agent(path=asma-furniturewala/gifgrep-agent/aixplain)

aiXplain empowers you to seamlessly build, customize, and deploy intelligent agents tailored to your specific needs. Whether you're creating standalone agents or advanced multi-agent systems, the platform provides a robust toolkit for integrating cutting-edge AI capabilities into your workflows. To learn more, visit [aiXplain](https://aixplain.com).
